In [ ]:
import os
import mlflow
import json
from mlflow import MlflowClient
from minio import Minio


In [ ]:
MLFLOW_TRACKING_URI = "http://mlflow-server.mlflow.svc.cluster.local:8080"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print("Mlflow URI:", mlflow.get_tracking_uri())

In [ ]:
# Find the latest run
experiment = client.get_experiment_by_name("openshift-ai-drift-blog-1")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=1
)
run = runs[0]
print("Run ID:", run.info.run_id)
print("Artifact URI:", run.info.artifact_uri)


In [ ]:
# Let's download serving artifact from MLflow
local_serving_model = mlflow.artifacts.download_artifacts(
    run_id=run.info.run_id,
    artifact_path="deployment/sklearn/iris_model.pkl",
    dst_path="serving-download"
)
print("Downloaded to:", local_serving_model)
import shutil
shutil.copy("models/iris_model.pkl","models/model.joblib")
    

In [ ]:
# Configure MinIO for OpenShift AI serving
MINIO_ENDPOINT = "minio-service.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"

client_minio = Minio(
    MINIO_ENDPOINT,
    access_key= MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)



In [ ]:
bucket = "mlops-models"
if not client_minio.bucket_exists(bucket):
    client_minio.make_bucket(bucket)

print("Bucket ready:", bucket)

In [ ]:
settings = {
    "name": "iris",
    "implementation": "mlserver_sklearn.SKLearnModel",
    "parameters": {
        "uri": "./iris_model.pkl"
    }
}
with open("models/model-settings.json", "w") as f:
    json.dump(settings, f, indent=2)

In [ ]:
# Upload model and model.json to serving path
object_name = "iris-model/kserve/iris_model.pkl"
client_minio.fput_object(
    bucket,
    object_name,
    local_serving_model
)
object_json_name = "iris-model/kserve/model-settings.json"
client_minio.fput_object(
    bucket,
    object_json_name,
    "models/model-settings.json"
)


In [ ]:
#Verify upload
objects = client_minio.list_objects(
    bucket,
    prefix="iris-model/kserve/",
    recursive=True
)
for obj in objects:
    print(obj.object_name, obj.size)